# Training time (V100): table + T-Maze Passive vs T

Reproduces the paper's training-time figure: a LaTeX table (left) and a T-Maze Passive panel (right).
The LaTeX wrapper is in `paper/training_time_figure_latex.md`.

**AMLT numbers.** Every model except Mamba ran on AMLT with one Tesla V100-SXM2-32GB per job. They come
from the W&B `info/duration_minute` histories:
- `duration_minute` restarts from 0 on every resume, so each history is split at the resets.
- rate = Σ(within-segment Δminutes) / Σ(Δepisodes), in minutes per 1k episodes.
- hours per run = rate × the episode budget.
- Only finished V100 runs count, excluding the user-reported bad runs. The report is the **median over seeds**,
  with no statistical outlier rule.

**Mamba** never ran on V100. It is estimated from short calibration runs on a rented 8×V100-SXM2-32GB node
(vast.ai, 2026-09-24):
- Mamba and two reference models ran with the same settings in the same wave (logs in
  `v100_calibration/`).
- estimate = geometric mean over the two references of (Mamba rate / reference rate) × the reference's AMLT
  median.
- The same ratio predicts one reference from the other, which checks the method's accuracy.

W&B histories are cached as JSON under `rl_results/training_time/`. Delete a file to download it again.

In [1]:
import json, re, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from paper_style import set_paper_style

set_paper_style()

## Config

In [2]:
ENTITY = "mate_research"
V100 = "Tesla V100-SXM2-32GB"
CACHE_DIR = Path("rl_results") / "training_time"
CALIB_DIR = Path("v100_calibration") / "timing_logs"
CALIB_START_EP = 512                 # skip torch.compile / warm-up; rate = slope over [512, end]
FIG_DIR = Path("figures")

# Run-name templates ({e}=env prefix, {m}=model tag, {s}=seed)
V2, V3 = "{e}_{m}_s{s}_v2", "{e}_{m}_s{s}_v3"
SEQ32 = "{e}_{m}_rl2x512_seq32_s{s}_v2"   # same setting as _v2 (used where _v2 is missing)
PRE = "{e}_{m}_rl2x512_s{s}"              # pre-v2 V100 runs (the _v2 ones of these cells ran on RTX 5090)

# Main environments: project -> (env prefix, episode budget, title, {method: (model tag, template)}, calib key).
# Every model at the environment's h/l (GPT-2 at l2 in l2 envs); Markov at 2h (as in the return curves).
MAIN = {
    "cheetah-vel":  ("cheetah", 50_000, "Cheetah-Vel", {
        "MATE": ("mate_h256_l1", V2), "GPT-2": ("gpt_h256_l1", V2), "LSTM": ("lstm_h256_l1", PRE),
        "SplAgger": ("splagger_h256_l1", PRE), "Markov (2h)": ("markov_h512_l1", V2)}, "cheetah_{m}_h256_l1"),
    "hopper-param": ("hopper", 100_000, "Hopper-Param", {
        "MATE": ("mate_h256_l2", V2), "GPT-2": ("gpt_h256_l2", V2), "LSTM": ("lstm_h256_l2", V2),
        "SplAgger": ("splagger_h256_l2", V2), "Markov (2h)": ("markov_h512_l2", V2)}, "hopper_{m}_h256_l2"),
    "ant-dir":      ("ant", 250_000, "Ant-Dir", {
        "MATE": ("mate_h512_l1", V2), "GPT-2": ("gpt_h512_l1", SEQ32), "LSTM": ("lstm_h512_l1", PRE),
        "SplAgger": ("splagger_h512_l1", PRE), "Markov (2h)": ("markov_h1024_l1", V2)}, "ant_{m}_h512_l1"),
    "walker-param": ("walker", 250_000, "Walker-Param", {
        "MATE": ("mate_h512_l2", V2), "GPT-2": ("gpt_h512_l2", PRE), "LSTM": ("lstm_h512_l2", SEQ32),
        "SplAgger": ("splagger_h512_l2", SEQ32), "Markov (2h)": ("markov_h1024_l2", V2)}, "walker_{m}_h512_l2"),
    "ML10":         ("ml10", 250_000, "ML10", {
        "MATE": ("mate_h256_l2", V3), "GPT-2": ("gpt_h256_l2", V3), "LSTM": ("lstm_h256_l2", V3),
        "SplAgger": ("splagger_h256_l2", V3), "Markov (2h)": ("markov_h512_l2", V3)}, "ml10_{m}_h256_l2"),
    "ML45":         ("ml45", 250_000, "ML45", {
        "MATE": ("mate_h512_l2", V3), "GPT-2": ("gpt_h512_l2", V3), "LSTM": ("lstm_h512_l2", V3),
        "SplAgger": ("splagger_h512_l2", V3), "Markov (2h)": ("markov_h1024_l2", V3)}, "ml45_{m}_h512_l2"),
}
MAIN_SEEDS = range(6)
BAD_RUNS = {"cheetah_mate_h256_l1_s1_v2", "ant_mate_h512_l1_s5_v2"}   # user-reported bad settings
MAIN_CALIB_REFS = ("MATE", "GPT-2")

# T-Maze Passive (_v4 sweep: DQN, critic 2x256, h128, l1), 100k episodes at every T
T_VALUES = [200, 400, 600, 800, 1000]
TMAZE_PROJECT = "tmaze_passive_T-{T}"
TMAZE_BUDGET = 100_000
TMAZE_RUNS = {
    "MATE": "passive_T{T}_mate_proj_h128_l1_critic2x256_s{s}_v4",   # MATE-proj: the only MATE variant in _v4
    "GPT-2": "passive_T{T}_gpt_h128_critic2x256_s{s}_v4",            # n_layer=1 (no _l in the name)
    "LSTM": "passive_T{T}_lstm_h128_l1_critic2x256_s{s}_v4",
    "SplAgger": "passive_T{T}_splagger_h128_l1_critic2x256_s{s}_v4",
}
TMAZE_SEEDS = range(5)
TMAZE_CALIB = "tmaze{T}_{m}_h128_l1"
TMAZE_CALIB_REFS = ("GPT-2", "LSTM")
CALIB_TAG = {"MATE": "mate", "GPT-2": "gpt", "LSTM": "lstm", "Mamba": "mamba"}

# Figure table (left) and T-Maze panel (right)
TABLE_ENVS = ["ant-dir", "walker-param", "ML45"]
TABLE_METHODS = ["MATE", "GPT-2", "LSTM", "SplAgger", "Mamba", "Markov (2h)"]
PANEL_METHODS = ["MATE", "GPT-2", "LSTM", "SplAgger", "Mamba"]   # no Oracle (user, 2026-09-25)
ESTIMATED = {"Mamba"}                                               # marked with * in the table
HOLLOW = set()                        # methods drawn with hollow markers in the panel (Mamba was, until 2026-09-25)

In [3]:
# ---- panel style (same palette / order as the main figure) ----
STYLE = {
    "MATE":     dict(color="#E41A1C", ls="-"),
    "GPT-2":    dict(color="#009E73", ls="-"),
    "LSTM":     dict(color="#0072B2", ls="-"),
    "SplAgger": dict(color="#AA3377", ls="-"),
    "Mamba":    dict(color="#E69F00", ls="-"),
}
LINE_W = 0.8                          # few points per line, so a bit heavier than the return curves (0.6)
EMPHASIS, EMPHASIS_SCALE = "MATE", 1.35
MARKER, MARKER_SIZE, MARKER_EDGE = "o", 2.6, 0.7
LEGEND_EDGE = "#d9d9d9"
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)
LEGEND_IN_PANEL = dict(loc="upper left", ncol=2)
TITLE, XLABEL, YLABEL = "T-Maze Passive", "Corridor Length $T$", "Training Time (h)"

# layout: the panel file is exactly PANEL_W x PANEL_H (half page = 0.48\textwidth, same as Vehicle Racing)
PANEL_W, PANEL_H = 2.64, 1.70
YTICK_RESERVE, XTICK_RESERVE_RIGHT, OUTER_PAD = "100", "1000", 0.02
OUT_STEM = "tmaze_passive_training_time"

## W&B histories (cached)

In [4]:
_api = None


def api():
    global _api
    if _api is None:
        import wandb
        _api = wandb.Api(timeout=180)
    return _api


def fetch_history(project, name):
    """Latest *finished* W&B run with this display name -> {rows: [(episodes, duration_minute)], gpu, id}.
    Cached as JSON; returns None when no finished run exists."""
    path = CACHE_DIR / project / f"{name}.json"
    if path.exists():
        d = json.loads(path.read_text())
        return d if d.get("rows") else None
    runs = [r for r in api().runs(f"{ENTITY}/{project}", filters={"display_name": name}) if r.state == "finished"]
    d = dict(name=name, project=project, rows=[])
    if runs:
        r = sorted(runs, key=lambda r: str(r.created_at))[-1]
        md = r.metadata or {}
        d.update(id=r.id, gpu=md.get("gpu"), cpu_count=md.get("cpu_count"),
                 rows=[(x.get("_step"), x.get("info/duration_minute"))
                       for x in r.scan_history(keys=["_step", "info/duration_minute"], page_size=2000)])
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(d))
    return d if d["rows"] else None


def restart_aware_rate(rows, min_rows=5):
    """Minutes per 1k episodes, summed over resume segments (duration_minute resets to 0 on resume)."""
    r = {int(e): float(m) for e, m in rows if e is not None and m is not None}   # last row per episode count
    ep = np.array(sorted(r)); dur = np.array([r[e] for e in ep])
    if len(ep) < min_rows:
        return np.nan
    dt = sp = 0.0
    for s in np.split(np.arange(len(ep)), np.where(np.diff(dur) < -0.5)[0] + 1):
        if len(s) >= min_rows:
            dt += dur[s[-1]] - dur[s[0]]; sp += ep[s[-1]] - ep[s[0]]
    return dt / sp * 1000 if sp else np.nan


jobs = []
for proj, (e, budget, title, methods, _) in MAIN.items():
    for method, (tag, tmpl) in methods.items():
        for s in MAIN_SEEDS:
            jobs.append(dict(suite="main", env=proj, method=method, seed=s, budget=budget, project=proj,
                             name=tmpl.format(e=e, m=tag, s=s)))
for T in T_VALUES:
    for method, tmpl in TMAZE_RUNS.items():
        for s in TMAZE_SEEDS:
            jobs.append(dict(suite="tmaze", env=T, method=method, seed=s, budget=TMAZE_BUDGET,
                             project=TMAZE_PROJECT.format(T=T), name=tmpl.format(T=T, s=s)))

with ThreadPoolExecutor(16) as ex:
    hists = list(ex.map(lambda j: fetch_history(j["project"], j["name"]), jobs))

records = []
for j, h in zip(jobs, hists):
    if h is None:
        continue
    rate = restart_aware_rate(h["rows"])
    records.append(dict(j, gpu=h.get("gpu"), rate=rate, hours=rate * j["budget"] / 1000 / 60,
                        bad=j["name"] in BAD_RUNS, v100=h.get("gpu") == V100))
runs = pd.DataFrame(records)
print(f"{len(runs)} finished runs; dropped: {int(runs.bad.sum())} user-reported bad, "
      f"{int((~runs.v100).sum())} non-V100 / unknown GPU")
runs[~runs.v100 | runs.bad][["suite", "env", "method", "name", "gpu"]]

265 finished runs; dropped: 2 user-reported bad, 1 non-V100 / unknown GPU


,suite,env,method,name,gpu
1,main,cheetah-vel,MATE,cheetah_mate_h256_l1_s1_v2,Tesla V100-SXM2-32GB
59,main,ant-dir,MATE,ant_mate_h512_l1_s5_v2,Tesla V100-SXM2-32GB
184,tmaze,200,SplAgger,passive_T200_splagger_h128_l1_critic2x256_s4_v4,None


## AMLT medians (V100 hours per run)

In [5]:
used = runs[runs.v100 & ~runs.bad & runs.rate.notna()]
agg = (used.groupby(["suite", "env", "method"])
       .agg(median_h=("hours", "median"), q1=("hours", lambda x: x.quantile(0.25)),
            q3=("hours", lambda x: x.quantile(0.75)), n=("hours", "size"), median_rate=("rate", "median"))
       .reset_index())
AMLT = {(r.suite, r.env, r.method): r.median_h for r in agg.itertuples()}
AMLT_RATE = {(r.suite, r.env, r.method): r.median_rate for r in agg.itertuples()}


def fmt(x):
    return "—" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.1f}"


main_tab = agg[agg.suite == "main"].pivot(index="method", columns="env", values="median_h")[list(MAIN)]
main_n = agg[agg.suite == "main"].pivot(index="method", columns="env", values="n")[list(MAIN)]
print("Main environments: median hours (n)")
display(main_tab.round(1).astype(str) + " (" + main_n.astype("Int64").astype(str) + ")")
tm_tab = agg[agg.suite == "tmaze"].pivot(index="method", columns="env", values="median_h")[T_VALUES]
print("T-Maze Passive: median hours per 100k episodes")
display(tm_tab.round(1))

Main environments: median hours (n)


env,cheetah-vel,hopper-param,ant-dir,walker-param,ML10,ML45
method,,,,,,
GPT-2,4.5 (5),9.5 (5),31.5 (5),33.2 (4),114.2 (6),198.4 (6)
LSTM,4.7 (5),12.0 (5),35.1 (4),34.8 (5),149.8 (6),231.7 (6)
MATE,4.2 (5),7.8 (6),22.1 (5),19.3 (6),91.7 (6),127.7 (6)
Markov (2h),3.8 (6),7.3 (6),27.0 (6),24.1 (6),96.6 (6),145.2 (6)
SplAgger,5.0 (5),11.0 (5),37.1 (4),35.7 (5),146.7 (6),223.9 (6)


T-Maze Passive: median hours per 100k episodes


env,200,400,600,800,1000
method,,,,,
GPT-2,6.2,17.2,35.5,63.2,99.9
LSTM,6.3,14.9,27.9,48.4,70.7
MATE,4.8,10.8,18.1,30.5,44.5
SplAgger,6.4,14.2,25.3,40.9,63.3


## Mamba: V100 calibration estimate

Rates in the calibration logs are the slope of wall time against the `Total rollouts` count over
`[CALIB_START_EP, end]`. The two halves of each window are printed as a check that the rate is steady.

In [6]:
_pat = re.compile(r"^(\d+\.\d+) Total rollouts:(\d+),")


def calib_rate(job):
    t, e = [], []
    for line in (CALIB_DIR / f"{job}.log").read_text(errors="replace").splitlines():
        m = _pat.match(line)
        if m:
            t.append(float(m[1])); e.append(int(m[2]))
    t, e = np.array(t), np.array(e)
    idx = np.where(e >= CALIB_START_EP)[0]
    slope = lambda i: np.polyfit(e[i], t[i], 1)[0] * 1000 / 60           # minutes per 1k episodes
    h = len(idx) // 2
    return slope(idx), slope(idx[:h + 1]), slope(idx[h:])


def mamba_estimate(suite, env, calib_job, refs):
    """Geometric mean over refs of (Mamba / ref) calibration ratio x ref's AMLT median, plus a cross-check."""
    rate = {m: calib_rate(calib_job.format(m=CALIB_TAG[m]))[0] for m in (*refs, "Mamba")}
    via = {r: rate["Mamba"] / rate[r] * AMLT[(suite, env, r)] for r in refs}
    a, b = refs
    check = rate[b] / rate[a] * AMLT[(suite, env, a)] / AMLT[(suite, env, b)] - 1   # predict b from a
    return math.sqrt(via[a] * via[b]), via, check, rate


rows = []
for proj, (*_, calib) in MAIN.items():
    est, via, check, rate = mamba_estimate("main", proj, calib, MAIN_CALIB_REFS)
    AMLT[("main", proj, "Mamba")] = est
    rows.append(dict(suite="main", env=proj, estimate_h=est, **{f"via {k}": v for k, v in via.items()},
                     check_err=check, **{f"calib {k}": v for k, v in rate.items()}))
for T in T_VALUES:
    est, via, check, rate = mamba_estimate("tmaze", T, TMAZE_CALIB.replace("{T}", str(T)), TMAZE_CALIB_REFS)
    AMLT[("tmaze", T, "Mamba")] = est
    rows.append(dict(suite="tmaze", env=T, estimate_h=est, **{f"via {k}": v for k, v in via.items()},
                     check_err=check, **{f"calib {k}": v for k, v in rate.items()}))
calib = pd.DataFrame(rows)
print(f"check (predict {MAIN_CALIB_REFS[1]} from {MAIN_CALIB_REFS[0]} / {TMAZE_CALIB_REFS[1]} from "
      f"{TMAZE_CALIB_REFS[0]}): mean |err| main {calib[calib.suite == 'main'].check_err.abs().mean():.0%}, "
      f"T-Maze {calib[calib.suite == 'tmaze'].check_err.abs().mean():.0%}")
calib.round(2)

check (predict GPT-2 from MATE / LSTM from GPT-2): mean |err| main 16%, T-Maze 14%


,suite,env,estimate_h,via MATE,via GPT-2,check_err,calib MATE,calib GPT-2,calib Mamba,via LSTM,calib LSTM
0,main,cheetah-vel,6.26,6.63,5.92,0.12,6.84,8.20,10.78,NaN,NaN
1,main,hopper-param,14.17,14.71,13.66,0.08,7.53,9.91,14.27,NaN,NaN
2,main,ant-dir,38.27,34.76,42.13,-0.17,7.34,8.63,11.53,NaN,NaN
3,main,walker-param,40.55,35.80,45.93,-0.22,7.53,10.09,13.94,NaN,NaN
4,main,ML10,146.33,155.34,137.84,0.13,18.84,26.46,31.93,NaN,NaN
5,main,ML45,193.57,216.84,172.79,0.25,25.52,49.75,43.33,NaN,NaN
6,tmaze,200,9.04,NaN,9.57,0.12,NaN,5.26,8.06,8.55,5.93
7,tmaze,400,23.59,NaN,27.44,0.35,NaN,10.08,16.05,20.28,11.79
8,tmaze,600,39.14,NaN,42.10,0.16,NaN,19.61,23.23,36.38,17.81
9,tmaze,800,62.64,NaN,61.61,-0.03,NaN,35.59,34.67,63.68,26.33


## Table (left half of the figure)

In [7]:
def table_latex(envs=TABLE_ENVS, methods=TABLE_METHODS):
    head = " & ".join(["Method"] + [MAIN[e][2] for e in envs]) + r" \\"
    best = {e: min(AMLT[("main", e, m)] for m in methods if not m.startswith("Markov")) for e in envs}
    lines = []
    for m in methods:
        label = {"MATE": "MATE (ours)", "Markov (2h)": "Markov ($2h$)"}.get(m, m) + ("$^{*}$" if m in ESTIMATED else "")
        cells = []
        for e in envs:
            v = AMLT.get(("main", e, m))
            cells.append(r"\textbf{%s}" % fmt(v) if v == best[e] else fmt(v))
        if m.startswith("Markov"):
            lines.append(r"      \midrule")
        lines.append(f"      {label:<18s} & " + " & ".join(cells) + r" \\")
    return "\n".join([r"    \begin{tabular}{l" + "c" * len(envs) + "}", r"      \toprule", "      " + head,
                      r"      \midrule", *lines, r"      \bottomrule", r"    \end{tabular}"])


tex = table_latex()
print(tex)
FIG_DIR.mkdir(parents=True, exist_ok=True)
(FIG_DIR / "training_time_table.tex").write_text(tex + "\n")

full = pd.DataFrame({MAIN[e][2]: {m: AMLT.get(("main", e, m)) for m in TABLE_METHODS} for e in MAIN})
print("\nAll six main environments (median V100 hours; Mamba estimated):")
full.round(1)

    \begin{tabular}{lccc}
      \toprule
      Method & Ant-Dir & Walker-Param & ML45 \\
      \midrule
      MATE (ours)        & \textbf{22.1} & \textbf{19.3} & \textbf{127.7} \\
      GPT-2              & 31.5 & 33.2 & 198.4 \\
      LSTM               & 35.1 & 34.8 & 231.7 \\
      SplAgger           & 37.1 & 35.7 & 223.9 \\
      Mamba$^{*}$        & 38.3 & 40.5 & 193.6 \\
      \midrule
      Markov ($2h$)      & 27.0 & 24.1 & 145.2 \\
      \bottomrule
    \end{tabular}

All six main environments (median V100 hours; Mamba estimated):


,Cheetah-Vel,Hopper-Param,Ant-Dir,Walker-Param,ML10,ML45
MATE,4.2,7.8,22.1,19.3,91.7,127.7
GPT-2,4.5,9.5,31.5,33.2,114.2,198.4
LSTM,4.7,12.0,35.1,34.8,149.8,231.7
SplAgger,5.0,11.0,37.1,35.7,146.7,223.9
Mamba,6.3,14.2,38.3,40.5,146.3,193.6
Markov (2h),3.8,7.3,27.0,24.1,96.6,145.2


## T-Maze Passive panel (right half of the figure)

In [8]:
def _text_extent(s, size, rotation=0, weight="normal"):
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def panel_margins():
    """(left, right, bottom, top) decoration space in inches, same rule as the main-figure notebooks."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, _ = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    xtick_h = _text_extent("0", rc["xtick.labelsize"])[1]
    left = (OUTER_PAD + ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
            + _text_extent(YLABEL, rc["axes.labelsize"], rotation=90)[0] + rc["axes.labelpad"] * pt)
    bottom = (OUTER_PAD + xtick_h + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
              + _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt)
    top = (OUTER_PAD + _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1]
           + rc["axes.titlepad"] * pt)
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    return left, right, bottom, top


def check_panel_fits(fig, ax):
    fig.canvas.draw()
    bb, fb = ax.get_tightbbox(fig.canvas.get_renderer()), fig.bbox
    over = {k: round(v / fig.dpi, 3) for k, v in
            {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}.items() if v > 0.5}
    if over:
        print(f"WARNING labels exceed the panel by {over} in")
    return fig.get_size_inches()


def style_legend_frame(legend):
    frame = legend.get_frame()
    frame.set_linewidth(0.4)
    n, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
    frame.set_path_effects([pe.SimplePatchShadow(offset=(size * k / n, -size * k / n), shadow_rgbFace="black",
                                                 alpha=alpha / n) for k in range(n, 0, -1)] + [pe.Normal()])


left, right, bottom, top = panel_margins()
fig = plt.figure(figsize=(PANEL_W, PANEL_H))
ax = fig.add_axes([left / PANEL_W, bottom / PANEL_H, 1 - (left + right) / PANEL_W, 1 - (bottom + top) / PANEL_H])
handles = []
for z, m in enumerate(PANEL_METHODS):
    st, emph, hollow = STYLE[m], m == EMPHASIS, m in HOLLOW
    lw = LINE_W * (EMPHASIS_SCALE if emph else 1.0)
    zorder = 3 + (1 if emph else 0) + z * 0.01
    y = [AMLT[("tmaze", T, m)] for T in T_VALUES]
    ax.plot(T_VALUES, y, color=st["color"], ls=st["ls"], lw=lw, zorder=zorder)
    ax.plot(T_VALUES, y, MARKER, ms=MARKER_SIZE, mew=MARKER_EDGE, color=st["color"],
            mfc="white" if hollow else st["color"], zorder=zorder + 0.5)
    handles.append(Line2D([], [], color=st["color"], lw=lw * 1.2, marker=MARKER, ms=MARKER_SIZE, mew=MARKER_EDGE,
                          mfc="white" if hollow else st["color"], label=m))
ax.set_xticks(T_VALUES)
ax.set_xlim(T_VALUES[0], T_VALUES[-1])
ax.set_ylim(0, None)
ax.set_title(TITLE)
ax.set_xlabel(XLABEL)
ax.set_ylabel(YLABEL)
ax.grid(True, ls="--", alpha=0.5)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
legend = ax.legend(handles=handles, frameon=True, fancybox=False, edgecolor=LEGEND_EDGE, facecolor="white",
                   framealpha=1.0, handlelength=1.8, handletextpad=0.4, columnspacing=0.8, borderpad=0.35,
                   labelspacing=0.25, borderaxespad=0.3, **LEGEND_IN_PANEL)
style_legend_frame(legend)
legend.set_zorder(10)
size = check_panel_fits(fig, ax)
fig.savefig(FIG_DIR / f"{OUT_STEM}.pdf")
fig.savefig(FIG_DIR / f"{OUT_STEM}.png", dpi=300)
print(f"saved {FIG_DIR / OUT_STEM}.pdf ({size[0]:.2f} x {size[1]:.2f} in)")
plt.show()

pd.DataFrame({m: [AMLT[("tmaze", T, m)] for T in T_VALUES] for m in PANEL_METHODS}, index=T_VALUES).T.round(1)

saved figures/tmaze_passive_training_time.pdf (2.64 x 1.70 in)


,200,400,600,800,1000
MATE,4.8,10.8,18.1,30.5,44.5
GPT-2,6.2,17.2,35.5,63.2,99.9
LSTM,6.3,14.9,27.9,48.4,70.7
SplAgger,6.4,14.2,25.3,40.9,63.3
Mamba,9.0,23.6,39.1,62.6,95.7
